# Stage 2 Data Preparation: Viet-ShareGPT-4o-Text-VQA + Viet-Localization-VQA

This notebook prepares the two public instruction datasets used in Stage 2. The output format is OpenAI-style multimodal chat JSONL.

In [ ]:
!pip install -q -U gdown datasets pillow pandas pyarrow matplotlib seaborn tqdm pyyaml loguru beautifulsoup4 requests openai python-dotenv kaggle

import json
import os
import random
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

try:
    from google.colab import userdata
except Exception:
    userdata = None

In [ ]:
REPO_URL = "https://github.com/duongtruongbinh/pretrain_vlm.git"
BRANCH = "heisenburger"
PROJECT_DIR = Path("/content/pretrain_vlm")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Project root:", Path.cwd())

HF_TOKEN = None
if userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("HF_TOKEN not set. Public datasets may still work; gated datasets require access approval.")

## Prepare instruction datasets

`sharegpt` prepares `5CD-AI/Viet-ShareGPT-4o-Text-VQA`. `5cd` prepares `5CD-AI/Viet-Localization-VQA` and may require Hugging Face access approval.

In [ ]:
subprocess.run(["python", "scripts/prepare_data.py", "sharegpt"], check=True)
subprocess.run(["python", "scripts/prepare_data.py", "5cd"], check=True)

## Inspect prepared splits

In [ ]:
def read_jsonl(path, limit=None):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def plot_counts(title, counts):
    series = pd.Series(counts, dtype="int64")
    ax = series.plot(kind="bar", figsize=(7, 4), color="#4C78A8")
    ax.set_title(title)
    ax.set_ylabel("rows")
    ax.bar_label(ax.containers[0])
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def show_image_grid(records, text_getter, n=6):
    samples = [r for r in records if Path(str(r.get("image", ""))).exists()]
    if not samples:
        print("No local image samples found.")
        return
    samples = random.sample(samples, min(n, len(samples)))
    cols = min(3, len(samples))
    rows = (len(samples) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 4.5 * rows))
    for i, record in enumerate(samples, start=1):
        image = Image.open(record["image"]).convert("RGB")
        ax = plt.subplot(rows, cols, i)
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(text_getter(record)[:120], fontsize=9)
    plt.tight_layout()
    plt.show()

stage2_paths = {
    "sharegpt_train": "data/viet-sharegpt-4o-text-vqa/train.jsonl",
    "sharegpt_val": "data/viet-sharegpt-4o-text-vqa/val.jsonl",
    "sharegpt_test": "data/viet-sharegpt-4o-text-vqa/test.jsonl",
    "localization_train": "data/viet-localization-vqa/train.jsonl",
    "localization_val": "data/viet-localization-vqa/val.jsonl",
    "localization_test": "data/viet-localization-vqa/test.jsonl",
}
counts = {name: count_jsonl(path) for name, path in stage2_paths.items()}
display(pd.DataFrame([{"split": k, "rows": v} for k, v in counts.items()]))
plot_counts("Stage 2 public instruction rows", counts)

## Visualize image and conversation samples

In [ ]:
records = []
for path in stage2_paths.values():
    records.extend(read_jsonl(path, limit=100))

def first_user_answer(record):
    messages = record.get("messages", [])
    user = next((m.get("content", "") for m in messages if m.get("role") == "user"), "")
    assistant = next((m.get("content", "") for m in messages if m.get("role") == "assistant"), "")
    return f"Q: {user}\nA: {assistant}"

show_image_grid(records, first_user_answer, n=6)